# **1. Q-Table**
- Q는 "현재 상태에서 이 행동을 했을 때 앞으로 얼마나 유리한가"를 나타내는 행동가치
- Q-Table은 이 값을 표 형태로 저장
- Q-Table은 정답표가 아님. 에이전트가 보상을 경험하며 채우고 수정하는 표

### Q-Table의 한계
- 로봇은 거리, 속도 , 방향 등 연속적인 숫자를 받음
- 상태를 세밀하게 구분하면 조합이 매우 큼. 예) 센서 3개를 각각 100단계로 구분하고 행동이 4개라면 100 * 100 * 100 * 4  = 4000000개의 Q값이 필요
- 상태마다 숫자를 저장한느 대신 상태 -> 신경망 -> 각 행동의 Q값이라는 관계를 학습
    - 처음 보는 상태도 비슷한 학습 경험을 활용해 Q깂을 예측 -> 함수 근사

> 신경망이 언제나 정확한 것은 아님. 학습 경험이 부족하거나 새로운 상황이 너무 다르면 잘못 예측할 수 있음

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn, optim
from collections import deque
import random

small = np.zeros((5, 2))
large_count = 100 ** 3 * 4
print('작은 미로 Q-Table 크기:', small.shape, '→', small.size, '개')
print('센서 3개를 100단계로 구분할 때:', f'{large_count:,}', '개')
print('float32로 저장할 때 약:', round(large_count * 4 / 1024**2, 2), 'MiB')

작은 미로 Q-Table 크기: (5, 2) → 10 개
센서 3개를 100단계로 구분할 때: 4,000,000 개
float32로 저장할 때 약: 15.26 MiB


In [2]:
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.xpu.is_available():
    DEVICE = torch.device('xpu')
else:
    DEVICE = torch.device('cpu')

print(DEVICE)

cpu


In [6]:
x = torch.arange(1, 6, dtype=torch.float32).unsqueeze(1)
y = x / 6.0
model = nn.Linear(1, 1)
optimizer = optim.SGD(model.parameters(), lr=0.01)

for epoch in range(1000):
    prediction = model(x)
    loss = ((prediction - y) ** 2).mean()
    optimizer.zero_grad() # 미분값 초기화
    loss.backward()
    optimizer.step()

In [7]:
print('w, b: ', model.weight.item(), model.bias.item()),
print('상태 2.5의 예측 가치: ', model(torch.tensor([[2.5]])).item())

w, b:  0.1640404611825943 0.009481383487582207
상태 2.5의 예측 가치:  0.4195825159549713


### DQN이 단순한 신경망 하나가 아닌 이유
- 신경망에 상태를 넣고  Q값만 예측할 수 없음
- 강화학습 데이터는 일반적인 사진 분류 데이터와 다르게 시간 순서대로 서로 비슷한 경험이 연속해서 발생
- 학습 목표를 만드는 데 사용하는 Q값도 신경망이 추정하므로 목표 자체가 계속 바뀜

1. Q-Network: 현재 상태를 보고 각 행동의 Q점수를 예측. 예) [왼쪽=12.1, 오른쪽=15.4]라면 오른쪽이 더 좋다고 판단
2. Experience Replay(경험 재생): 상태, 행동, 보상, 다음 상태, 종료 여부
